<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/MUSIC-ACESTEP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Music — ACE-Step 1.5

Makes songs (with vocals and lyrics) or instrumentals from a style description, with **ACE-Step 1.5**. Works on the
**free T4** and better on an **L4 / A100** (cell 1 picks the XL model and a bigger song planner there).

1. **Runtime → Change runtime type → T4 GPU** (or L4 / A100).
2. Run **cell 1** (install + downloads, ~10–20 GB), **cell 2** (saves the server program) and **cell 3** (starts it and
   prints the `READY:` URL).
3. On the website open the **Music** tab and paste the URL.

On a T4 ACE-Step runs in fp32 (no bf16 there), so songs take longer. Cell 4 shows the server log and memory use.
Don't run another GPU notebook at the same time.

In [ ]:
# Cell 1: install ComfyUI, check the GPU and download ACE-Step 1.5 (music)
%cd /content
import os, json, subprocess
!git clone -q https://github.com/comfyanonymous/ComfyUI /content/ComfyUI || true
!cd /content/ComfyUI && git pull -q || true
!cd /content/ComfyUI && pip install -q -r requirements.txt
!pip install -q fastapi uvicorn requests psutil
!apt install -qqy aria2 ffmpeg > /dev/null

# ---- GPU
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip().split("\n")[0]
except FileNotFoundError:  # nvidia-smi only exists when a GPU is attached
    gpu = ""
if "," not in gpu:
    raise RuntimeError(
        "This Colab runtime has no GPU, and music generation needs one.\n"
        "Fix: Runtime → Change runtime type → a GPU → Save, then run the cells again.\n"
        "If Colab can't connect to a GPU, your GPU time or compute units have run out for now (check Resources).")
gpu_name, vram_mib = [x.strip() for x in gpu.rsplit(",", 1)]
GPU = {"name": gpu_name, "vram_gib": round(int(vram_mib) / 1024, 1)}

# ---- Model: ACE-Step 1.5 (Hugging Face: Comfy-Org/ACE-Step-1.5), turbo versions for speed.
#      Picked for the GPU: T4 -> standard model + 1.7B song planner; L4 -> XL model; A100 -> XL + 4B planner.
#      (A T4 has no bf16, so ACE-Step runs in fp32 there: it works, just slower.)
HF = "https://huggingface.co/Comfy-Org/ACE-Step-1.5/resolve/main/split_files"
big = GPU["vram_gib"] >= 20
huge = GPU["vram_gib"] >= 38
model_file = "acestep_v1.5_xl_turbo_bf16.safetensors" if big else "acestep_v1.5_turbo.safetensors"
planner_file = "qwen_4b_ace15.safetensors" if huge else "qwen_1.7b_ace15.safetensors"
FILES = {
    "model": (model_file, f"{HF}/diffusion_models/{model_file}", "diffusion_models"),
    "text_encoder": ("qwen_0.6b_ace15.safetensors", f"{HF}/text_encoders/qwen_0.6b_ace15.safetensors", "text_encoders"),
    "planner": (planner_file, f"{HF}/text_encoders/{planner_file}", "text_encoders"),
    "vae": ("ace_1.5_vae.safetensors", f"{HF}/vae/ace_1.5_vae.safetensors", "vae"),
}
DEFAULTS = {"steps": 8, "cfg": 1.0, "sampler": "euler", "scheduler": "simple",  # turbo: few steps, no CFG
            "planner_cfg": 2.0, "max_seconds": 240 if big else 180}
LABEL = f"ACE-Step 1.5 {'XL ' if big else ''}turbo + {'4B' if huge else '1.7B'} planner"

# ---- Download (resumes if interrupted) and write the config the server reads
for key, (name, url, folder) in FILES.items():
    print(f"Downloading {name}...")
    subprocess.run(["aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M", url,
                    "-d", f"/content/ComfyUI/models/{folder}", "-o", name], check=True)
json.dump({"label": LABEL, "gpu": GPU, "defaults": DEFAULTS, "files": {k: v[0] for k, v in FILES.items()}},
          open("/content/music_config.json", "w"), indent=1)
print(f"GPU: {GPU['name']} ({GPU['vram_gib']} GB) -> {LABEL}. Now run cells 2 and 3.")

In [ ]:
%%writefile /content/music_server.py
# Cell 2: the music server, saved as its own program (/content/music_server.py).
# Cell 3 runs it separately from this notebook: if a song runs out of memory only this program dies,
# and cell 3 restarts it automatically (same URL) instead of the whole Colab session crashing.
# Running this cell only saves the file; edit it here and re-run cells 2 and 3 to apply changes.
import os
import sys
import json

os.chdir("/content/ComfyUI")
sys.path.insert(0, "/content/ComfyUI")

CRASH_FILE = "/content/server_crash.txt"
CONFIG = json.load(open("/content/music_config.json"))  # written by cell 1: model files, GPU, defaults


def last_crash_message():
    """Set by cell 3 when the server crashed and was restarted, so the website can say why a job vanished."""
    try:
        return open(CRASH_FILE).read().strip()
    except OSError:
        return ""


import gc
import time
import uuid
import random
import logging
import warnings
import threading
import traceback
import subprocess

import numpy as np
import torch

logging.getLogger("root").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

import nodes
import comfy.utils
from comfy_extras import nodes_ace, nodes_audio

M = nodes.NODE_CLASS_MAPPINGS
UNETLoader, DualCLIPLoader, VAELoader = M["UNETLoader"](), M["DualCLIPLoader"](), M["VAELoader"]()
KSampler, ConditioningZeroOut = M["KSampler"](), M["ConditioningZeroOut"]()
F = CONFIG["files"]
D = CONFIG["defaults"]

print(f"Loading {CONFIG['label']}...", flush=True)
with torch.inference_mode():
    model = UNETLoader.load_unet(F["model"], "default")[0]
    clip = DualCLIPLoader.load_clip(F["text_encoder"], F["planner"], "ace")[0]  # lyrics encoder + song planner LM
    vae = VAELoader.load_vae(F["vae"])[0]
print("Models loaded.", flush=True)

KEYS = [f"{root} {q}" for q in ("major", "minor") for root in
        ["C", "C#", "Db", "D", "D#", "Eb", "E", "F", "F#", "Gb", "G", "G#", "Ab", "A", "A#", "Bb", "B"]]
LANGUAGES = ["en", "es", "fr", "de", "it", "pt", "nl", "pl", "ru", "uk", "tr", "ar", "hi", "id", "ja", "ko", "zh",
             "yue", "vi", "th", "tl", "sv", "no", "da", "fi", "cs", "ro", "hu", "el", "he", "fa", "ms", "unknown"]
AUDIO_DIR = "/content/music"
os.makedirs(AUDIO_DIR, exist_ok=True)


def save_mp3(audio, path):
    """audio: {"waveform": [B, C, T] float, "sample_rate": int} -> 192 kbps MP3."""
    wave = audio["waveform"][0].float().cpu().numpy()  # [C, T]
    channels, sr = wave.shape[0], int(audio["sample_rate"])
    pcm = np.ascontiguousarray(wave.T.clip(-1, 1).astype("<f4"))
    proc = subprocess.Popen(["ffmpeg", "-y", "-loglevel", "error", "-f", "f32le", "-ar", str(sr), "-ac", str(channels),
                             "-i", "-", "-b:a", "192k", path], stdin=subprocess.PIPE)
    proc.stdin.write(pcm.tobytes())
    proc.stdin.close()
    if proc.wait() != 0:
        raise RuntimeError("ffmpeg could not write the MP3")
    return round(wave.shape[1] / sr, 1)


# ---------------------------------------------------------------- jobs
JOBS = {}
gpu_lock = threading.Lock()
current_job = {"id": None}


def progress_hook(value, total, *args, **kwargs):
    job = JOBS.get(current_job["id"])
    if job and total:
        job["step"], job["total_steps"] = int(value), int(total)


comfy.utils.set_progress_bar_global_hook(progress_hook)


def new_job(**extra):
    now = time.time()
    for jid in [j for j, v in JOBS.items() if v["status"] in ("done", "error") and now - v["created"] > 3600]:
        JOBS.pop(jid, None)
    job_id = uuid.uuid4().hex[:12]
    JOBS[job_id] = {"status": "queued", "created": now, "error": None, "stage": "queued", "step": 0,
                    "total_steps": 0, **extra}
    return job_id


def memory_report():
    try:
        import psutil
        vm = psutil.virtual_memory()
        ram = f"RAM {(vm.total - vm.available) / 1e9:.1f}/{vm.total / 1e9:.1f} GB"
    except Exception:
        ram = "RAM ?"
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        return f"{ram}, VRAM {(total - free) / 1e9:.1f}/{total / 1e9:.1f} GB"
    return ram


def run_song(job_id, req):
    job = JOBS[job_id]
    with gpu_lock:
        current_job["id"] = job_id
        job["status"] = "running"
        t0 = time.time()
        try:
            seconds = max(5.0, min(float(req.duration), D["max_seconds"]))
            seed = req.seed if req.seed and req.seed > 0 else random.randint(1, 2**63 - 1)
            lyrics = "[Instrumental]" if req.instrumental or not req.lyrics.strip() else req.lyrics
            key = req.key if req.key in KEYS else "C major"
            language = req.language if req.language in LANGUAGES else "en"
            sig = str(req.timesignature) if str(req.timesignature) in ("2", "3", "4", "6") else "4"
            steps = max(1, min(int(req.steps or D["steps"]), 100))
            job.update(seconds=seconds, seed=seed)
            print(f"[{job_id}] {seconds:.0f} s, {req.bpm} bpm, {key}, {steps} steps, planner {req.planner}", flush=True)
            gc.collect()
            torch.cuda.empty_cache()
            with torch.inference_mode():
                job["stage"] = "planning the song" if req.planner else "reading the style and lyrics"
                positive = nodes_ace.TextEncodeAceStepAudio15.execute(
                    clip, req.tags, lyrics, seed % (2**63), int(req.bpm), seconds, sig, language, key,
                    bool(req.planner), D["planner_cfg"], 0.85, 0.9, 0, 0.0).args[0]
                negative = ConditioningZeroOut.zero_out(positive)[0]
                latent = nodes_ace.EmptyAceStep15LatentAudio.execute(seconds, 1).args[0]
                job["stage"] = "making the music"
                samples = KSampler.sample(model, seed, steps, D["cfg"], D["sampler"], D["scheduler"],
                                          positive, negative, latent, 1.0)[0]
                job["stage"] = "decoding the audio"
                audio = nodes_audio.VAEDecodeAudio.execute(vae, samples).args[0]
            job["stage"] = "saving the MP3"
            length = save_mp3(audio, os.path.join(AUDIO_DIR, f"{job_id}.mp3"))
            job.update(status="done", stage="done", audio=f"/audio/{job_id}.mp3", seconds=length,
                       took=round(time.time() - t0))
            print(f"[{job_id}] done in {job['took']} s. {memory_report()}", flush=True)
        except Exception as e:
            traceback.print_exc()
            job.update(status="error", error=str(e))
        finally:
            current_job["id"] = None
            gc.collect()
            torch.cuda.empty_cache()


# ---------------------------------------------------------------- API for the website
from typing import Optional
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True,
                   allow_methods=["*"], allow_headers=["*"])


class SongRequest(BaseModel):
    tags: str = ""               # style: genre, instruments, mood, voice
    lyrics: str = ""             # with [verse] / [chorus] / [bridge] sections
    instrumental: bool = False
    duration: float = 60
    bpm: int = 120
    key: str = "C major"
    timesignature: str = "4"
    language: str = "en"
    planner: bool = True         # the LM plans the song structure first (better, a bit slower)
    steps: Optional[int] = None
    seed: int = -1


@app.get("/")
def root():
    return {"status": "ok", "mode": "music", "model": CONFIG["label"]}


@app.get("/music_info")
def music_info():
    return {"mode": "music", "label": CONFIG["label"], "gpu": CONFIG["gpu"], "max_seconds": D["max_seconds"],
            "steps": D["steps"], "keys": KEYS, "languages": LANGUAGES}


@app.post("/generate_music")
def generate_music(req: SongRequest):
    job_id = new_job()
    threading.Thread(target=run_song, args=(job_id, req), daemon=True).start()
    return {"job_id": job_id, "status": "queued"}


@app.get("/job/{job_id}")
def get_job(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        return {"status": "error", "error": last_crash_message() or "Unknown job (the server restarted)"}
    return {"job_id": job_id, **{k: v for k, v in job.items() if k != "created"}}


@app.get("/audio/{name}")
def get_audio(name: str):
    path = os.path.join(AUDIO_DIR, os.path.basename(name))
    if not name.endswith(".mp3") or not os.path.exists(path):
        return {"error": "not found"}
    return FileResponse(path, media_type="audio/mpeg", filename=name)


if __name__ == "__main__":
    import uvicorn
    print("Server listening on port 8000.", flush=True)
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

In [ ]:
# Cell 3: start the generation server as a separate program, plus the Cloudflare tunnel.
# If the server crashes (e.g. runs out of memory) only that program dies: it's restarted automatically
# with the same URL, and this notebook keeps running. Re-running this cell restarts the server with the
# current code from cell 2. To stop everything: Runtime → Restart session.
import os, re, sys, time, subprocess, threading, urllib.request

SERVER_SCRIPT = "/content/music_server.py"
SERVER_LOG = "/content/music_server.log"
CRASH_FILE = "/content/server_crash.txt"
PORT = 8000
CLOUDFLARED = "/usr/local/bin/cloudflared"


def server_up():
    try:
        return urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=3).status == 200
    except Exception:
        return False


def log_tail(n=15):
    try:
        return "".join(open(SERVER_LOG, errors="replace").readlines()[-n:])
    except OSError:
        return ""


class Supervisor:
    """Runs the server program and restarts it when it crashes (gives up after 4 crashes in 10 minutes)."""

    def __init__(self):
        self.proc, self.thread, self.stopping, self.gave_up, self.crashes = None, None, False, False, []

    def _loop(self):
        while not self.stopping:
            with open(SERVER_LOG, "a") as log:
                log.write(f"\n===== {time.strftime('%H:%M:%S')} starting the server =====\n")
                log.flush()
                self.proc = subprocess.Popen([sys.executable, "-u", SERVER_SCRIPT], cwd="/content/ComfyUI",
                                             stdout=log, stderr=subprocess.STDOUT,
                                             env={**os.environ, "PYTHONUNBUFFERED": "1"},
                                             start_new_session=True)  # stopping a notebook cell must not stop the server
                code = self.proc.wait()
            if self.stopping:
                break
            why = ("ran out of memory and was stopped by the system" if code in (-9, 137)
                   else f"crashed (exit code {code})")
            self.crashes = [t for t in self.crashes if time.time() - t < 600] + [time.time()]
            if len(self.crashes) >= 4:
                msg = (f"The generation server {why} 4 times in 10 minutes, so it was not restarted. "
                       f"Check the log (cell 4), then run cell 3 again.")
                self.gave_up = True
            else:
                msg = (f"The generation server {why} at {time.strftime('%H:%M')} and was restarted "
                       f"(models reload in 1-3 minutes). Try a smaller size, fewer pictures or batch 1.")
            with open(CRASH_FILE, "w") as fh:
                fh.write(msg)
            with open(SERVER_LOG, "a") as log:
                log.write(f"\n!!!!! {msg}\n")
            if self.gave_up:
                break
            time.sleep(3)

    def start(self):
        self.stopping, self.gave_up = False, False
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()

    def stop(self):
        self.stopping = True
        if self.proc and self.proc.poll() is None:
            self.proc.terminate()
            try:
                self.proc.wait(30)
            except subprocess.TimeoutExpired:
                self.proc.kill()
        if self.thread:
            self.thread.join(35)


# (Re)start the server program
if "SUPERVISOR" in globals():
    print("Restarting the server with the current code...")
    try:
        SUPERVISOR.stop()
    except Exception:
        pass
if os.path.exists(CRASH_FILE):
    os.remove(CRASH_FILE)
SUPERVISOR = Supervisor()
SUPERVISOR.start()

print("Loading ACE-Step in the server program (1-3 minutes)...")
t0 = time.time()
while not server_up():
    if SUPERVISOR.gave_up or time.time() - t0 > 900:
        print("The server did not start. Last lines of its log:\n" + log_tail(30))
        raise RuntimeError("Server failed to start — see the log above")
    time.sleep(5)
print(f"Server is up after {time.time() - t0:.0f}s.")

# Cloudflare tunnel (started once; the URL stays the same when the server restarts).
# If it stopped (e.g. the runtime hiccuped), a new one is started — with a NEW URL to paste into the website.
if globals().get("TUNNEL_URL") and (globals().get("TUNNEL_PROC") is None or TUNNEL_PROC.poll() is not None):
    print("The old tunnel stopped, so the old URL no longer works — starting a new one (new URL below).")
    TUNNEL_URL = None
if not globals().get("TUNNEL_URL"):
    subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
    if not os.path.exists(CLOUDFLARED):
        subprocess.run(["wget", "-q", "-O", CLOUDFLARED,
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
        os.chmod(CLOUDFLARED, 0o755)
    TUNNEL_PROC = subprocess.Popen([CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                   start_new_session=True)  # survives stopping a notebook cell
    t0 = time.time()
    while time.time() - t0 < 60 and not globals().get("TUNNEL_URL"):
        line = TUNNEL_PROC.stdout.readline()
        if not line:
            break
        m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
        if m:
            TUNNEL_URL = m.group(0)
    threading.Thread(target=lambda: [None for _ in TUNNEL_PROC.stdout], daemon=True).start()  # keep the pipe drained

if globals().get("TUNNEL_URL"):
    print("=" * 60)
    print(f"READY: {TUNNEL_URL}")
    print("Paste this into the website: Music tab → Music server URL.")
    print("This cell is done; the server keeps running in the background.")
    print("Cell 4 shows its log and memory use. To stop it: Runtime → Restart session.")
    print("=" * 60)
else:
    print("Could not start the Cloudflare tunnel. Run this cell again.")

In [ ]:
# Cell 4 (optional): the server's recent log and memory use. Run it again to refresh.
import subprocess
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f"System RAM: {(vm.total - vm.available) / 1e9:.1f} / {vm.total / 1e9:.1f} GB")
except Exception:
    pass
gpu = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
if gpu:
    print(f"GPU memory: {gpu}")
sup = globals().get("SUPERVISOR")
if sup is not None:
    running = sup.proc is not None and sup.proc.poll() is None
    print("Server program:", "running" if running else ("stopped — run cell 3" if sup.gave_up else "starting / restarting"))
    print(f"Crashes in the last 10 minutes: {len([t for t in sup.crashes if __import__('time').time() - t < 600])}")
tp = globals().get("TUNNEL_PROC")
if globals().get("TUNNEL_URL"):
    alive = tp is not None and tp.poll() is None
    print(f"Website URL: {TUNNEL_URL}", "(working)" if alive else "— STOPPED: run cell 3 again and paste its new URL into the website")
print("-" * 60)
try:
    print("".join(open("/content/music_server.log", errors="replace").readlines()[-40:]))
except OSError:
    print("No log yet — run cell 3 first.")